# Exploratory Data Analysis

**Purpose** : notebook focuses on a first pass at assessing experimental data on a subject and group basis via EDA. The relevant sections are:

1. Trial exposure + data quality
2. Learning signal
3. Task transfer
4. Individual differences

Specific tests applied in each section are described below the respective headings - interpretations follow.

In [38]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats

In [39]:
MIN_TRIALS = 5

df_clean = pd.read_csv('../data/processed/clean_data.csv')

trials = df_clean[df_clean['watch_only'] == 0].copy()
trials['pressed'] = trials['pressed'].astype('int64')
trials['outcome'] = trials['outcome'].astype('int64')
trials = trials.rename(columns={'prop_SC': 'sc_prop', 'prop_OC': 'oc_prop', 'prop_R': 'r_prop'})

_t1 = df_clean[df_clean['task'] == 1].groupby('subject_id').first().reset_index()
_press = trials.groupby('subject_id')['pressed'].mean().rename('overall_press_rate')
subject_meta = (
    _t1[['subject_id', 'group', 'prop_SC', 'prop_OC', 'prop_R', 'completion_min']]
    .rename(columns={'prop_SC': 'sc_prop', 'prop_OC': 'oc_prop', 'prop_R': 'r_prop'})
    .merge(_press, on='subject_id')
)

press_rates = (
    trials
    .groupby(['subject_id', 'group', 'sc_prop', 'task', 'block', 'trial_role'])
    .agg(
        n_trials  =('pressed', 'count'),
        n_pressed =('pressed', 'sum'),
        press_rate=('pressed', 'mean'),
        mean_rt   =('rt',      'mean'),
    )
    .reset_index()
)

learning_curves = (
    trials
    .groupby(['task', 'block', 'trial_n', 'trial_role'])['pressed']
    .mean()
    .reset_index()
    .rename(columns={'pressed': 'mean_press_rate'})
)

pr = press_rates[press_rates['n_trials'] >= MIN_TRIALS].copy()

print(f'trials:          {trials.shape}')
print(f'subject_meta:    {subject_meta.shape}')
print(f'press_rates:     {press_rates.shape}')
print(f'learning_curves: {learning_curves.shape}')


trials:          (11200, 20)
subject_meta:    (70, 7)
press_rates:     (828, 10)
learning_curves: (480, 5)


## 1. Trial exposure and data quality

- Trial counts per subject per role per block per task
- SC property value distribution across subjects
- Group A vs B trial distribution comparison
- Confirm everyone got correct trial counts globally


In [40]:
# Global trial counts
print(f"Active trials (watch-only Block 1 trials removed) : {len(trials):,}")
print(f"Subjects : {trials['subject_id'].nunique()}")
print()
print()

# Trial counts per subject × role × block 
count_stats = (
    press_rates
    .groupby('trial_role')['n_trials']
    .describe()[['min', '25%', '50%', '75%', 'max']]
    .astype(int)
)
print("Trials per subject per role per block :")
print(count_stats)

block2 = press_rates[press_rates['block'] == 2]
block2_low = block2[block2['n_trials'] < MIN_TRIALS]
print(f"\nCells below MIN_TRIALS={MIN_TRIALS} in Block 2: {len(block2_low)} / {len(block2)} ({100*len(block2_low)/len(block2):.1f}%)")
print()
print()

# Group × SC property summary
print("Subjects by group × SC property:")
print(pd.crosstab(subject_meta['group'], subject_meta['sc_prop'], margins=True))


Active trials (watch-only Block 1 trials removed) : 11,200
Subjects : 70


Trials per subject per role per block :
            min  25%  50%  75%  max
trial_role                         
OC            1    3    5    9   16
R             1    3    6    9   17
SC            8   14   26   42   51

Cells below MIN_TRIALS=5 in Block 2: 16 / 420 (3.8%)


Subjects by group × SC property:
sc_prop  colour  size  velocity  All
group                               
A            12     9        18   39
B            10    16         5   31
All          22    25        23   70


Trial counts and assignments are globally correct.
Active trial counts, subject counts, and the distribution of trials per role per block are overall correct : the bulk of low-count cells are attributed to Block 3 (only 20 trials, sparse OC and R assignments regardless). Block 2 minimal cells only make up 3.8% of total cells, which is expected under random assignment.
    
The one note to make is on SC property assignments by group : `size` assignments were ~50% lower for Group A than for Group B, with the reverse holding for `velocity` (assignment to Group B < Group A). `Colour`remains balanced across the groups - it was the property with the most variation overall (changing values between T1 and T2). 
Overall, the distribution of assignments remains acceptable for the sake of this analysis, and does not disrupt results.

## 2. Learning signal

- Rolling mean SC press rate over block 2 trials (approx. learning curve shape)
- SC outcome rate conditional on pressing - proxy for 'accuracy' in the tasks
- Average press rate per role per block, separately per task and per group
- SC vs OC rate relationship: do they move proportionately, or does SC selectively diverge? Plot distributions on same axes.

In [41]:
# Rolling mean SC press rate

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, task in zip(axes, [1, 2]):
    sc = (
        learning_curves
        .query("task == @task and block == 2 and trial_role == 'SC'")
        .sort_values('trial_n')
    )
    ax.plot(sc['trial_n'], sc['mean_press_rate'], alpha=0.3, color='steelblue', label='raw')
    ax.plot(sc['trial_n'], sc['mean_press_rate'].rolling(5, center=True).mean(),
            color='steelblue', linewidth=2, label='5-trial rolling mean')
    ax.set(title=f'Task {task} — SC learning curve (block 2)',
           xlabel='Trial', ylabel='Mean press rate')
    ax.legend()
plt.tight_layout()
plt.savefig('../output/EDA_sc_learning_curve.png', dpi=150, bbox_inches='tight')
print()


# SC outcome rate conditional on pressing

sc_pressed = trials[(trials['trial_role'] == 'SC') & (trials['pressed'] == 1)]
outcome_by_block = (
    sc_pressed
    .groupby(['task', 'block'])['outcome']
    .agg(mean_outcome='mean', n='count')
    .reset_index()
)
print("SC outcome rate given pressed (design check — expect ~0.95):")
print(outcome_by_block.to_string(index=False))
print()


# Average press rate per role per block, by task and group

summary = (
    pr.groupby(['group', 'task', 'block', 'trial_role'])['press_rate']
    .mean()
    .reset_index()
)
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True)
for row, group in enumerate(['A', 'B']):
    for col, task in enumerate([1, 2]):
        ax = axes[row][col]
        subset = summary.query("group == @group and task == @task")
        for role, colour in [('SC', 'steelblue'), ('OC', 'darkorange'), ('R', 'grey')]:
            d = subset[subset['trial_role'] == role]
            ax.plot(d['block'], d['press_rate'], marker='o', color=colour, label=role)
        ax.set(title=f'Group {group}, Task {task}',
               xlabel='Block', ylabel='Mean press rate', xticks=[1, 2, 3])
        ax.legend()
plt.tight_layout()
plt.savefig('../output/EDA_press_rate_by_role_block.png', dpi=150, bbox_inches='tight')


# SC vs OC press rate distributions on same axes

b2 = pr[(pr['block'] == 2)]
sc_rates = b2[b2['trial_role'] == 'SC'].set_index(['subject_id', 'task'])['press_rate']
oc_rates = b2[b2['trial_role'] == 'OC'].set_index(['subject_id', 'task'])['press_rate']

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True, sharex=True)
for ax, task in zip(axes, [1, 2]):
    sc = sc_rates.xs(task, level='task')
    oc = oc_rates.xs(task, level='task')
    ax.hist(sc, bins=15, alpha=0.5, color='steelblue', label='SC')
    ax.hist(oc, bins=15, alpha=0.5, color='darkorange', label='OC')
    ax.axvline(sc.mean(), color='steelblue', linestyle='--', linewidth=1.5, label=f'SC mean={sc.mean():.2f}')
    ax.axvline(oc.mean(), color='darkorange', linestyle='--', linewidth=1.5, label=f'OC mean={oc.mean():.2f}')
    ax.set(title=f'Task {task} — block 2 press rate distributions',
           xlabel='Press rate', ylabel='Subjects')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('../output/EDA_sc_oc_distributions.png', dpi=150, bbox_inches='tight')


print("Rolling mean SC press rate figure saved to output/: 'EDA_sc_learning_curve.png'")
print("Average press rate x role x block figures saved to output/: 'EDA_press_rate_by_role_block.png'")

print("SC vs. OC press rate distributions figure saved to output/: 'EDA _sc_oc_distributions.png'")



SC outcome rate given pressed (design check — expect ~0.95):
 task  block  mean_outcome    n
    1      2      0.947011 1472
    1      3      0.946535  505
    2      2      0.957232 1286
    2      3      0.962376  505

Rolling mean SC press rate figure saved to output/: 'EDA_sc_learning_curve.png'
Average press rate x role x block figures saved to output/: 'EDA_press_rate_by_role_block.png'
SC vs. OC press rate distributions figure saved to output/: 'EDA _sc_oc_distributions.png'


The press rate x role x block shows a clear relationship between properties and press rate.
- OC, in every case, has a press rate > 0.5, that continues increasing as tasks develop. It shows strong learning across all cases; always above chance.
- SC > R in every case, but by a thin margin. It hovers around 0.5 - chance - in all 4 scenarios.
- R press rates decline sharply in every scenario, with the only attenuation of this effect being seen in T1 Group A. This coincides with the highest increase in OC press rates.

Group assignments appear to influence press rates in 1 scenario : Group B, Task 1. By design, Group B subjects were exposed to random outcomes in their Block 1 trials - the OC property was not active, resulting in no clear causal reversals (as SC was not possible to observe, the trials being watch-only). 
It is interesting to observe that in those circumstances, OC and SC are pressed at the same rate : 0.5. OC is not only neutral, but decreases over T1. This indicates that observing causal structure in training blocks may intervene with SC learning, and provide learning signals that override others during Block 2 and 3. This observation extends to Group B's T2, in which SC and OC continue to be very tightly correlated and close in rate, increasing at the same pace. 


Rolling mean SC press rates, across both Tasks, show a clear increasing trend - this supports the claim that SC-learning is accumulated over time. 

Finally, property-specific press-rate distributions (SC vs. OC in Tasks 1 vs. 2) show clear differences in shape and skew:
- OC press rate mean (= 0.64) in Task 1 is clearly driven by a majority of subjects pressing at near-1 press rates. This can be seen through the left-hand skew of the distribution
- SC, in Task 1, seems normally distributed, with mean = 0.50 and no clear bias
- In Task 2, both distributions change significantly, appearing more flat : SC mean = 0.43, driven by a clear majority of subjects pressing < 0.5. OC mean = 0.58, driven by a relatively flat distribution (most subjects lying between 0.4 - 1.0). 

## 3. Task transfer

- SC Block 2 press rates : Task 1 vs Task 2 (paired t-test)
- SC Block 2 press rates : first 10 trials, T1 vs. T2 (paired t-test)
- OC T1 property press rate in task 2 (property swap should drive it to R level if transfer is property-specific, rather than general)
- Transfer index distribution across subjects
- Value-specific vs property-type transfer: identify subjects where SC positive value is same vs different across tasks, compare transfer indices

In [47]:
# SC Block 2 press rates : T1 vs T2

sc_b2 = (
    pr.query("block == 2 and trial_role == 'SC'")
    [['subject_id', 'group', 'sc_prop', 'task', 'press_rate']]
    .pivot_table(index=['subject_id', 'group', 'sc_prop'], columns='task', values='press_rate')
    .rename(columns={1: 'task1', 2: 'task2'})
    .reset_index()
    .dropna()
)
print(f"SC block 2 press rate (N={len(sc_b2)})")
print(f"Task 1: mean={sc_b2['task1'].mean():.3f}, sd={sc_b2['task1'].std():.3f}")
print(f"Task 2: mean={sc_b2['task2'].mean():.3f}, sd={sc_b2['task2'].std():.3f}")
t, p = stats.ttest_rel(sc_b2['task2'], sc_b2['task1'])
print(f"Paired t (T2 vs T1): t={t:.3f}, p={p:.4f}")

early = (
    trials[trials['block'] == 2]
    .sort_values(['subject_id', 'task', 'trial_n'])
    .groupby(['subject_id', 'task'])
    .head(10)
)
sc_early = (
    early[early['trial_role'] == 'SC']
    .groupby(['subject_id', 'task'])['pressed']
    .agg(n='count', press_rate='mean')
    .reset_index()
    .query("n >= 3")
    .pivot_table(index='subject_id', columns='task', values='press_rate')
    .rename(columns={1: 'task1_early', 2: 'task2_early'})
    .reset_index()
    .dropna()
)
print(f"\nEarly trials — first 10 of block 2 (N={len(sc_early)})")
print(f"Task 1 early: mean={sc_early['task1_early'].mean():.3f}")
print(f"Task 2 early: mean={sc_early['task2_early'].mean():.3f}")
t_e, p_e = stats.ttest_rel(sc_early['task2_early'], sc_early['task1_early'])
print(f"Paired t: t={t_e:.3f}, p={p_e:.4f}")
print()


# Former OC property press rate in Task 2

former_oc = pr.query("task==2 and block==2 and trial_role=='R'")[['subject_id','press_rate']].rename(columns={'press_rate':'former_oc_as_r_t2'})
oc_t1     = pr.query("task==1 and block==2 and trial_role=='OC'")[['subject_id','press_rate']].rename(columns={'press_rate':'oc_t1'})
r_t1      = pr.query("task==1 and block==2 and trial_role=='R'")[['subject_id','press_rate']].rename(columns={'press_rate':'r_t1'})
new_oc_t2 = pr.query("task==2 and block==2 and trial_role=='OC'")[['subject_id','press_rate']].rename(columns={'press_rate':'new_oc_t2'})
oc_df = former_oc.merge(oc_t1,'inner','subject_id').merge(r_t1,'inner','subject_id').merge(new_oc_t2,'left','subject_id')

print(f"\nOC property across tasks (N={len(oc_df)})")
print(f"R T1 baseline: {oc_df['r_t1'].mean():.3f}")
print(f"OC T1 baseline: {oc_df['oc_t1'].mean():.3f}")
print(f"Former OC (new R) in T2: {oc_df['former_oc_as_r_t2'].mean():.3f}")
print(f"Former R (new OC) in T2: {oc_df['new_oc_t2'].mean():.3f}")
t_drop, p_drop = stats.ttest_rel(oc_df['former_oc_as_r_t2'], oc_df['oc_t1'])
t_base, p_base = stats.ttest_rel(oc_df['former_oc_as_r_t2'], oc_df['r_t1'])
print()
print(f"Former OC T2 vs OC T1: t={t_drop:.3f}, p={p_drop:.4f}")
print(f"Former OC T2 vs R baseline: t={t_base:.3f}, p={p_base:.4f}")
print()


# Transfer index distribution

sc_b2['transfer_idx'] = sc_b2['task2'] - sc_b2['task1']
ti = sc_b2['transfer_idx']
print(f"\nTransfer index (T2 SC − T1 SC, block 2, N={len(ti)})")
print(f"Mean={ti.mean():.3f}  Median={ti.median():.3f}  Std={ti.std():.3f}")
print(f"% positive: {(ti > 0).mean()*100:.1f}%   % negative: {(ti < 0).mean()*100:.1f}%")
t_ti, p_ti = stats.ttest_1samp(ti, 0)
print()


# Transfer by SC property

prop_order = ['colour', 'size', 'velocity']
prop_colours = {'colour': 'mediumorchid', 'size': 'steelblue', 'velocity': 'seagreen'}
prop_markers = {'colour': 'o', 'size': 's', 'velocity': '^'}
transfer_by_prop = (
    sc_b2.groupby('sc_prop')['transfer_idx']
    .agg(n='count', mean='mean', sd='std', sem='sem')
    .reindex(prop_order)
)
print(f"\nTransfer index by SC property")
print(transfer_by_prop.round(3))
f_prop, p_prop = stats.f_oneway(
    *[sc_b2.loc[sc_b2['sc_prop'] == prop, 'transfer_idx'] for prop in prop_order]
)
print(f"Transfer index differs by SC property: F={f_prop:.3f}, p={p_prop:.4f}")


# Figures

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]
for prop in prop_order:
    d = sc_b2[sc_b2['sc_prop'] == prop]
    ax.scatter(
        d['task1'], d['task2'],
        color=prop_colours[prop], marker=prop_markers[prop],
        alpha=0.6, label=prop, s=40
    )
ax.plot([0,1],[0,1],'k--',linewidth=1,label='no change')
ax.set(xlabel='Task 1 SC press rate', ylabel='Task 2 SC press rate',
       title='SC Block 2 press rates : Task 1 vs Task 2', xlim=(-0.05,1.05), ylim=(-0.05,1.05))
ax.legend(fontsize=8)

ax = axes[1]
x = np.arange(len(prop_order))
means = transfer_by_prop.loc[prop_order, 'mean']
sems = transfer_by_prop.loc[prop_order, 'sem']
bars = ax.bar(
    x, means, yerr=sems, capsize=4,
    color=[prop_colours[prop] for prop in prop_order],
    alpha=0.75, edgecolor='white'
)
for i, prop in enumerate(prop_order):
    d = sc_b2.loc[sc_b2['sc_prop'] == prop, 'transfer_idx']
    jitter = np.linspace(-0.12, 0.12, len(d)) if len(d) > 1 else np.array([0])
    ax.scatter(
        np.full(len(d), i) + jitter, d,
        color='black', alpha=0.35, s=18, linewidth=0
    )
    ax.text(i, means.loc[prop], f"{means.loc[prop]:+.2f}", ha='center', va='bottom' if means.loc[prop] >= 0 else 'top', fontsize=8)
ax.axhline(0, color='black', linestyle='--', linewidth=1)
ax.set(
    xticks=x,
    xticklabels=prop_order,
    ylabel='Transfer index (T2 − T1, ±SEM)',
    title='Transfer index by SC property'
)

ax = axes[2]
lbls  = ['R\n(T1 baseline)', 'OC\n(T1)', 'Former OC\nas R (T2)', 'New OC\n(T2)']
means = [oc_df['r_t1'].mean(), oc_df['oc_t1'].mean(), oc_df['former_oc_as_r_t2'].mean(), oc_df['new_oc_t2'].mean()]
sems  = [oc_df[c].sem() for c in ['r_t1','oc_t1','former_oc_as_r_t2','new_oc_t2']]
bars  = ax.bar(lbls, means, color=['grey','darkorange','darkorange','darkorange'],
               alpha=0.75, edgecolor='white', yerr=sems, capsize=4)
bars[2].set_alpha(0.35)
ax.axhline(oc_df['r_t1'].mean(), color='grey', linestyle='--', linewidth=1)
ax.set(ylabel='Mean press rate (±SEM)', title='OC / R swap : press rates across tasks', ylim=(0,1.05))

plt.tight_layout()
plt.savefig('../output/EDA_task_transfer.png', dpi=150, bbox_inches='tight')
print()
print("Task transfer figures saved to output/: 'EDA_task_transfer.png'")

SC block 2 press rate (N=70)
Task 1: mean=0.501, sd=0.254
Task 2: mean=0.433, sd=0.273
Paired t (T2 vs T1): t=-2.180, p=0.0327

Early trials — first 10 of block 2 (N=70)
Task 1 early: mean=0.454
Task 2 early: mean=0.398
Paired t: t=-1.180, p=0.2420


OC property across tasks (N=63)
R T1 baseline: 0.408
OC T1 baseline: 0.623
Former OC (new R) in T2: 0.377
Former R (new OC) in T2: 0.572

Former OC T2 vs OC T1: t=-5.292, p=0.0000
Former OC T2 vs R baseline: t=-0.832, p=0.4086


Transfer index (T2 SC − T1 SC, block 2, N=70)
Mean=-0.067  Median=-0.049  Std=0.259
% positive: 37.1%   % negative: 61.4%


Transfer index by SC property
           n   mean     sd    sem
sc_prop                          
colour    22 -0.082  0.277  0.059
size      25  0.016  0.239  0.048
velocity  23 -0.144  0.245  0.051
Transfer index differs by SC property: F=2.431, p=0.0957

Task transfer figures saved to output/: 'EDA_task_transfer.png'


Both tests set to determine whether SC press rates transferred (T1 vs. T2, mean values, or early trial values) report conclusive negative results:
- Mean press rates (paired t-test) : t = -2.18, p = 0.0327
- Early trial press rates (paired t-test) : t = -1.18, p = 0.242        

This indicates that SC - property-indexed - learning does not transfer across tasks, falsifying our driving hypothesis. Hypothesis tests and further analysis in `03_hypothesis_tests.ipynb` sheds more light on this.       

The property swap (OC -> R, and R -> OC) in Task 2 shows a clear reversal and re-adjustment pattern : 
- OC T1 baseline is 0.623. This drops to 0.377 in Task 2, rapidly adjusting to the new reversal base rates
- R T1 baseline is 0.408. This increases to 0.572 in Task 2, also rapidly adjusting to the new reversal base rates

The 2 t-tests run confirm this. The difference in strength of evidence may be attributable to the uncertainty associated with OC and R properties, respectively - this would explain the sharper drop in OC -> R, than in R -> OC.        
OC T2 - OC T1 : t = -5.292, p = 0.000.     
OC T2 - R baseline : t = -0.832, p = 0.4086

Finally, the overall SC transfer index (distribution overall, and by property) confirms that SC transfer does not occur reliably (either due to the signal being too weak regardless, or simply not transferring). However, the property-specific view shows nuance : `size` shows different measures than `colour`and `velocity`, with mean = 0.016. Colour and velocity means are both negative - the related plot shows marked variability in the transfer indexes (velocity being the most visible, with clear negative values across the board. Colour shows more scatter in the distribution).
I ran a simple ANOVA to test whether this was statistically significant : results are negative.

## 4. Individual differences

- Between-subject variability in SC-R gap
- SC property type as moderator (colour vs size vs velocity)
- Group A vs B SC press rate distributions on same plot
- Block 3: is SC press rate stable, rising, or decaying per subject? Classify subjects into maintainers vs diffusers.

In [59]:
# SC-R gap per subject

sc_r = (
    pr.query("block == 2 and trial_role in ['SC','R']")
    .pivot_table(index=['subject_id','group','sc_prop','task'], columns='trial_role', values='press_rate')
    .reset_index()
    .dropna(subset=['SC','R'])
)
sc_r['sc_r_gap'] = sc_r['SC'] - sc_r['R']

print("SC-R gap (block 2)")
print(sc_r.groupby('task')['sc_r_gap'].describe()[['count','mean','std','min','max']].round(3))
print()


# SC property type as moderator

print("\nSC-R gap by SC property type and task")
print(sc_r.groupby(['sc_prop','task'])['sc_r_gap'].agg(n='count', mean='mean', sd='std').round(3))
print()
for task in [1, 2]:
    groups = [sc_r.query("sc_prop==@p and task==@task")['sc_r_gap'].dropna() for p in ['colour','size','velocity']]
    f, pf = stats.f_oneway(*groups)
    print(f"One-way ANOVA task {task}: F={f:.3f}, p={pf:.4f}")
print()


# Group A vs B SC press rate

print("\nGroup A vs B SC block 2 press rate by group and task")
grp = pr.query("block==2 and trial_role=='SC'")[['subject_id','group','task','press_rate']]
print(grp.groupby(['group','task'])['press_rate'].agg(n='count', mean='mean', sd='std').round(3))
print()
for task in [1, 2]:
    a = grp.query("group=='A' and task==@task")['press_rate']
    b = grp.query("group=='B' and task==@task")['press_rate']
    t_g, p_g = stats.ttest_ind(a, b)
    
    print(f"Group A vs B, task {task}: t={t_g:.3f}, p={p_g:.4f}")
print()


# Maintainers vs diffusers

b2_sc = pr.query("block==2 and trial_role=='SC'")[['subject_id','task','press_rate']].rename(columns={'press_rate':'b2'})
b3_sc = pr.query("block==3 and trial_role=='SC'")[['subject_id','task','press_rate']].rename(columns={'press_rate':'b3'})
stab  = b2_sc.merge(b3_sc, on=['subject_id','task']).dropna()
stab['change'] = stab['b3'] - stab['b2']

THR = 0.10
stab['cls'] = 'stability'
stab.loc[stab['change'] >  THR, 'cls'] = 'increase'
stab.loc[stab['change'] < -THR, 'cls'] = 'diffusion'

print()
print(f"Increase, stability, diffusion — Block 3 SC change (threshold ±{THR})")
print(stab.groupby(['task','cls']).size().unstack(fill_value=0))
print()
print("\n  Mean B3−B2 change:")
print(stab.groupby('task')['change'].agg(mean='mean', sd='std', n='count').round(3))
print()

# Figures

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

ax = axes[0][0]
for task, col in [(1,'steelblue'),(2,'darkorange')]:
    d = sc_r[sc_r['task']==task]['sc_r_gap']
    ax.hist(d, bins=15, alpha=0.5, color=col, edgecolor='white', label=f'Task {task}')
ax.axvline(0, color='black', linestyle='--', linewidth=1)
ax.set(xlabel='SC press rate − R press rate', ylabel='Subjects',
       title='SC-R gap distribution (block 2)')
ax.legend()

ax = axes[0][1]
x, w = np.arange(3), 0.35
for i, (task, col) in enumerate([(1,'steelblue'),(2,'darkorange')]):
    means = [sc_r.query("sc_prop==@p and task==@task")['sc_r_gap'].mean() for p in ['colour','size','velocity']]
    sems  = [sc_r.query("sc_prop==@p and task==@task")['sc_r_gap'].sem()  for p in ['colour','size','velocity']]
    ax.bar(x + i*w, means, w, color=col, alpha=0.75, label=f'Task {task}', yerr=sems, capsize=3)
ax.set(xticks=x+w/2, xticklabels=['colour','size','velocity'],
       ylabel='SC-R gap (±SEM)', title='SC-R gap by SC property type')
ax.axhline(0, color='black', linewidth=0.8)
ax.legend()

ax = axes[1][0]
for task, ls in [(1,'-'),(2,'--')]:
    for grp_val, col in [('A','steelblue'),('B','darkorange')]:
        d = pr.query("block==2 and trial_role=='SC' and group==@grp_val and task==@task")['press_rate']
        ax.hist(d, bins=12, alpha=0.25, color=col, edgecolor='white', histtype='stepfilled')
        ax.axvline(d.mean(), color=col, linestyle=ls, linewidth=2,
                   label=f'Grp {grp_val} T{task} ({d.mean():.2f})')
ax.set(xlabel='SC press rate (block 2)', ylabel='Subjects', title='Group A vs B SC press rate')
ax.legend(fontsize=7)

ax = axes[1][1]
for cls, col in [('maintainer','seagreen'),('stable','steelblue'),('diffuser','tomato')]:
    d = stab[stab['cls']==cls]
    ax.scatter(d['b2'], d['b3'], color=col, alpha=0.55, s=40, label=f'{cls} (n={len(d)})')
ax.plot([0,1],[0,1],'k--',linewidth=1)
ax.set(xlabel='Block 2 SC press rate', ylabel='Block 3 SC press rate',
       title='Block 3 stability (both tasks)', xlim=(-0.05,1.05), ylim=(-0.05,1.05))
ax.legend()

plt.tight_layout()
plt.savefig('../output/EDA_individual_differences.png', dpi=150, bbox_inches='tight')
print()
print("Individual differences figures saved to output/: 'EDA_individual_differences.png'")

SC-R gap (block 2)
      count   mean    std    min    max
task                                   
1      69.0  0.085  0.307 -1.000  0.808
2      66.0  0.049  0.273 -0.521  0.818


SC-R gap by SC property type and task
                n   mean     sd
sc_prop  task                  
colour   1     22  0.073  0.276
         2     21 -0.013  0.327
size     1     25  0.088  0.199
         2     25  0.132  0.200
velocity 1     22  0.094  0.428
         2     20  0.011  0.278

One-way ANOVA task 1: F=0.027, p=0.9738
One-way ANOVA task 2: F=1.926, p=0.1542


Group A vs B SC block 2 press rate by group and task
             n   mean     sd
group task                  
A     1     39  0.483  0.273
      2     39  0.381  0.263
B     1     31  0.522  0.231
      2     31  0.499  0.275

Group A vs B, task 1: t=-0.633, p=0.5291
Group A vs B, task 2: t=-1.827, p=0.0720


Increase, stability, diffusion — Block 3 SC change (threshold ±0.1)
cls   diffusion  increase  stability
task                     

In regards to individual differences in SC press-rates:
- Differences in SC press-rates vs. R press-rates are stable across tasks, with small differences in mean (0.085 vs. 0.049). The magnitude of effect sizes aren't sufficient to imply a real gap between SC press-rates across tasks.
- When viewed through property-type, this relationship continues to hold : both ANOVA measures taken to verify intra-task property differences in SC-R press rates are statistically insignificant.
- As regards group-specific SC press rates, across tasks, small effects can be observed - again, however, these are statistically insignificant when measured through paired t-tests. 

In regards to SC stability across different blocks:
- Subjects were split into different categories: *maintainers*, *diffusers*, and *stable*. These measured the direction and magnitude of subjects' press rates over time [-0.1 < X < +0.1].
- Most subjects held stable or increased their press rates across Block 2 and Block 3 (approx. 2/3)
- Task 2 saw slightly more subjects increasing their press rates in proportion, though this is likely attributable to overall lower press rates observed in T2.

Overall, no significant relationships or effect sizes were observed across SC and R press-rates, or across blocks.